# ДЗ №2. Структуры данных: массив, список, хеш

Курс «Алгоритмы и структуры данных» · по материалам лекции 2

**Фамилия, группа:** *(впишите сюда)*

## Как работать с этим ноутбуком

- Демо-ячейки уже исполнены — выводы видны сразу. Прогоните их сами: Shift+Enter, сверху вниз.
- Ячейки **«### Ответ»** — ваши: пишите в них, заглушку затрите.
- Пометка `# <-- можно менять здесь` — приглашение поиграть с числами.
- Нужен только Python 3.9+, без пакетов и интернета.
- Точные числа у вас будут другими — важен порядок величин, не миллисекунды.

## Шпаргалка с лекции

| Структура | Сильная сторона | Слабая сторона |
|---|---|---|
| Массив | чтение по индексу O(1) | вставка/удаление O(n) |
| Связанный список | вставка/удаление O(1) | чтение O(n) |
| Стек / очередь | простой порядок обработки | доступ только к концам |
| Хеш-таблица | поиск/вставка/удаление O(1) в среднем | нет порядка, худший случай O(n) |

**Амортизированная стоимость** — средняя стоимость одной операции, когда
редкие дорогие операции размазываются на много дешёвых. Динамический массив
(Python `list`) удваивается при переполнении: вставка в конец — O(1) в среднем.

---

## Задача 1. Журнал транзакций

Финансовое приложение: тысячи транзакций в день **дописываются в конец**
журнала, чтение записей редкое. Что взять — массив или связанный список?

Сначала рассуждение, потом эксперимент: ячейка ниже сравнивает вставку
в конец и в начало растущего списка — порциями, чтобы было видно,
как ведёт себя время каждой порции.

In [1]:
import time

N = 20_000
CHUNK = 5_000

def time_chunks(insert_at_end):
    data = []
    times = []
    for _ in range(N // CHUNK):
        t0 = time.perf_counter()
        for i in range(CHUNK):
            if insert_at_end:
                data.append(i)       # в конец
            else:
                data.insert(0, i)    # в начало
        times.append(time.perf_counter() - t0)
    return times

tail = time_chunks(True)
head = time_chunks(False)

for k, (a, b) in enumerate(zip(tail, head), 1):
    print(f"порция {k}: в конец {a*1000:6.1f} мс, в начало {b*1000:7.1f} мс")

порция 1: в конец    1.6 мс, в начало    30.1 мс
порция 2: в конец    1.2 мс, в начало    90.5 мс
порция 3: в конец    0.7 мс, в начало   146.0 мс
порция 4: в конец    1.4 мс, в начало   162.3 мс


### Ответ (задача 1)

*(массив или список — и почему. Что эксперимент говорит про вставку в конец?
Что — про вставку в начало?)*

---

## Задача 2. Топ-10 злодеев

Сайт ведёт рейтинг из 1 000 000 записей. Пользователь листает кнопкой Next —
читает записи **подряд, начиная с первой**. Массив или связанный список?

Маленький эксперимент про чтение «подряд с начала»: сколько стоит
прочитать первые 10 записей из миллиона.

In [2]:
million = list(range(1_000_000))

t0 = time.perf_counter()
top10 = million[:10]                      # первые 10, подряд с начала
t_first10 = time.perf_counter() - t0

t0 = time.perf_counter()
everything = million[:]                   # а так — пройти весь миллион
t_all = time.perf_counter() - t0

print(f"прочитать первые 10:    {t_first10 * 1_000_000:8.1f} мкс")
print(f"прочитать весь миллион: {t_all * 1000:8.1f} мс")
print(f"разница: {t_all / t_first10:,.0f} раз")

прочитать первые 10:       149.9 мкс
прочитать весь миллион:      8.3 мс
разница: 55 раз


### Ответ (задача 2)

*(какая структура достаточна и почему: какой доступ нужен — случайный
или последовательный? Принимается любой выбор с аргументом)*

---

## Задача 3. Хеш-таблица, заполненная на 5%

Хеш-таблица на 10 000 ячеек, в ней 50 записей — заполнение 0.05.
Что с ней не так и что делать?

Это задача-рассуждение, но потыкать всё равно интересно: ниже — игрушечная
хеш-таблица с открытой адресацией, как на лекции.

In [3]:
import sys

class SimpleHashTable:
    """Игрушечная хеш-таблица с открытой адресацией — как на лекции."""

    def __init__(self, capacity):
        self.capacity = capacity
        self.cells = [None] * capacity
        self.size = 0

    def put(self, key):
        i = hash(key) % self.capacity
        while self.cells[i] is not None:
            i = (i + 1) % self.capacity        # коллизия — шаг вперёд
        self.cells[i] = key
        self.size += 1

    def load_factor(self):
        return self.size / self.capacity

t = SimpleHashTable(10_000)
for k in range(50):
    t.put(f"запись-{k}")

print(f"записей: {t.size}, ячеек: {t.capacity:,}")
print(f"заполнение: {t.load_factor():.3f}")
print(f"память под ячейки: {sys.getsizeof(t.cells) / 1024:.0f} КБ")
print(f"полезно занято: {t.size * sys.getsizeof(None) / 1024:.1f} КБ")

записей: 50, ячеек: 10,000
заполнение: 0.005
память под ячейки: 78 КБ
полезно занято: 0.8 КБ


### Ответ (задача 3)

*(что не так с почти пустой таблицей и что делать: как выбрать размер?)*

---

## Задача 4 (со звёздочкой). Голосовал ли избиратель?

По списку из 1 000 000 фамилий нужно много раз проверять: голосовал ли
избиратель с такой-то фамилией? Какая структура данных подходит
и какой выигрыш против линейного поиска?

Сначала — гипотеза письменно. Потом замер: для скорости возьмём 300 000
фамилий (полный миллион тоже ок, просто ячейка будет работать дольше —
попробуйте оба варианта).

In [4]:
import random

random.seed(42)
N = 300_000                       # <-- можно менять здесь (1_000_000 — тоже ок)
surnames = [f"фамилия-{i}" for i in range(N)]

voters_list = surnames[:]         # копия-список
voters_set = set(surnames)        # копия-хеш-таблица

probe = surnames[-1]              # последняя: худший случай для линейного поиска

t0 = time.perf_counter()
found = probe in voters_list      # линейный поиск
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
found = probe in voters_set       # хеш-таблица
t_set = time.perf_counter() - t0

print(f"линейный поиск: {t_list * 1000:8.2f} мс")
print(f"хеш-таблица:    {t_set * 1000:8.4f} мс")
print(f"выигрыш: {t_list / t_set:,.0f} раз")

линейный поиск:    15.79 мс
хеш-таблица:      0.1523 мс
выигрыш: 104 раз


### Ответ (задача 4)

*(какая структура и почему; какой выигрыш даёт замер — и сколько это будет
на полном миллионе, если экстраполировать ваши числа?)*

---

## Сдача

Чек-лист перед отправкой:

- имя и группа вписаны в титул;
- демо-ячейки прогнаны, выводы на месте;
- все ячейки «### Ответ» заполнены;
- файл сохранён и отправлен преподавателю (.ipynb).

Если какая-то ячейка не запускается или что-то пошло не так — напишите
прямо в этой ячейке, что произошло: это нормально.